# K4 — Ablation tìm kiếm A1, A2 và A5

Notebook đọc lại log thô, kiểm tra bảng tổng hợp và dựng hình. Không chạy lại
các trận chỉ để vẽ biểu đồ. Chạy các thí nghiệm từ `Connect_Four/` trước:

```sh
PYTHONPATH=src python scripts/run_search_ablations.py --experiment a1
PYTHONPATH=src python scripts/run_search_ablations.py --experiment a5
PYTHONPATH=src python scripts/run_search_ablations.py --experiment a2
```

Giữ nguyên thuật toán, dùng bộ vị trí chung; khai cuộc đối đầu được ghép cặp
đổi bên. Seed của tác tử ngẫu nhiên trong ván i là `20261104 + i` và được dùng
lại ở mỗi mức ngân sách. Chạy tuần tự để tránh các thí nghiệm tranh CPU.


In [1]:
from pathlib import Path
import os
import sys

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
elif (root / "Connect_Four").is_dir():
    root = root / "Connect_Four"
assert (root / "src/connect_four").is_dir(), "Chạy từ repo, Connect_Four hoặc notebooks"
os.chdir(root)
sys.path[:0] = [str(root / "src"), str(root / "scripts")]
from analyze_search_ablations import analyze

tables = analyze()
print("Đã đối chiếu bảng với log thô và sinh lại hình, report_numbers.csv.")


Đã đối chiếu bảng với log thô và sinh lại hình, report_numbers.csv.


## A1 — Cắt tỉa và thứ tự duyệt

Bộ kiểm thử có cả thế cờ đã hòa; thế kết thúc được ghi `excluded_terminal`,
không đo một lệnh chọn nước không hợp lệ. Mỗi thế còn chơi được được đo năm
lần ở bốn cấu hình và ba độ sâu. Các cấu hình phải cho cùng **giá trị minimax**;
cột chọn có thể khác khi nhiều nước cùng điểm tối ưu.


In [2]:
for r in tables["a1"]:
    print(f"depth={r['depth']} pruning={r['pruning']} ordering={r['ordering']}: "
          f"{r['mean_nodes']:.1f} nodes, {r['mean_ms']:.3f} ms, p95={r['p95_ms']:.3f} ms")


depth=4 pruning=1 ordering=1: 235.0 nodes, 6.532 ms, p95=31.578 ms
depth=4 pruning=1 ordering=0: 401.3 nodes, 11.974 ms, p95=35.549 ms
depth=4 pruning=0 ordering=1: 1609.8 nodes, 56.691 ms, p95=98.267 ms
depth=4 pruning=0 ordering=0: 1609.8 nodes, 55.133 ms, p95=98.629 ms
depth=5 pruning=1 ordering=1: 1042.8 nodes, 31.365 ms, p95=141.155 ms
depth=5 pruning=1 ordering=0: 1598.1 nodes, 48.706 ms, p95=159.614 ms
depth=5 pruning=0 ordering=1: 9714.4 nodes, 321.222 ms, p95=689.120 ms
depth=5 pruning=0 ordering=0: 9714.4 nodes, 319.806 ms, p95=684.304 ms
depth=6 pruning=1 ordering=1: 2697.1 nodes, 79.571 ms, p95=411.416 ms
depth=6 pruning=1 ordering=0: 5042.0 nodes, 153.838 ms, p95=631.507 ms
depth=6 pruning=0 ordering=1: 58678.0 nodes, 1984.020 ms, p95=4773.420 ms
depth=6 pruning=0 ordering=0: 58678.0 nodes, 1964.872 ms, p95=4909.615 ms


![A1: nút và thời gian](../results/figures/ablation_a1_search.png)


## A2 — Ngân sách MCTS

Mỗi mức 100/500/1000/2000/5000 mô phỏng đánh 100 ván gặp Alpha-Beta độ sâu 4.
Đây là khảo sát ngân sách mô phỏng, **không phải** giải đấu cùng thời gian mỗi nước.
Khoảng tin cậy chính dùng bootstrap trên 50 cặp khai cuộc (10.000 lần lấy mẫu),
giữ hai lượt đổi bên trong cùng một đơn vị lấy mẫu. Wilson của tỷ lệ thắng chỉ
là số tham khảo vì giả định ván độc lập không hoàn toàn đúng với thiết kế cặp.


In [3]:
for r in tables["a2"]:
    print(f"{r['simulations']} simulations: {r['wins']}W/{r['draws']}D/{r['losses']}L, "
          f"score={r['score']:.3f}, paired 95% CI=[{r['score_ci_low']:.3f}, {r['score_ci_high']:.3f}], "
          f"mean={r['mean_ms']:.3f} ms, p95={r['p95_ms']:.3f} ms")


100 simulations: 12W/3D/85L, score=0.135, paired 95% CI=[0.070, 0.215], mean=2.544 ms, p95=3.965 ms
500 simulations: 34W/4D/62L, score=0.360, paired 95% CI=[0.280, 0.445], mean=12.218 ms, p95=19.530 ms
1000 simulations: 57W/5D/38L, score=0.595, paired 95% CI=[0.510, 0.680], mean=24.721 ms, p95=40.470 ms
2000 simulations: 60W/5D/35L, score=0.625, paired 95% CI=[0.540, 0.705], mean=49.084 ms, p95=80.130 ms
5000 simulations: 72W/4D/24L, score=0.740, paired 95% CI=[0.660, 0.815], mean=126.824 ms, p95=208.365 ms


![A2: ngân sách MCTS](../results/figures/ablation_a2_search.png)


## A5 — Độ sâu Alpha-Beta

Độ sâu 2–6, 100 ván/mức trước Heuristic, giữ cùng khai cuộc và quy tắc seed.
Phần đo thế cờ lặp ba lần, giữ cắt tỉa và sắp xếp bật. Thời gian trên thế cờ và
thời gian trong đối đầu là hai mẫu khác nhau, không gộp thành một số.


In [4]:
for r in tables["a5"]:
    if r["phase"] == "games":
        print(f"depth={r['depth']}: {r['wins']}W/{r['draws']}D/{r['losses']}L, "
              f"score={r['score']:.3f}, CI=[{r['score_ci_low']:.3f}, {r['score_ci_high']:.3f}], "
              f"mean={r['mean_ms']:.3f} ms, p95={r['p95_ms']:.3f} ms")
    else:
        print(f"depth={r['depth']}, positions: {r['mean_nodes']:.1f} nodes, {r['mean_ms']:.3f} ms")


depth=2, positions: 23.0 nodes, 0.576 ms
depth=3, positions: 93.0 nodes, 2.575 ms
depth=4, positions: 235.0 nodes, 6.117 ms
depth=5, positions: 1042.8 nodes, 29.653 ms
depth=6, positions: 2697.1 nodes, 73.408 ms
depth=2: 72W/9D/19L, score=0.765, CI=[0.690, 0.835], mean=0.748 ms, p95=1.443 ms
depth=3: 57W/6D/37L, score=0.600, CI=[0.515, 0.685], mean=3.784 ms, p95=8.022 ms
depth=4: 79W/5D/16L, score=0.815, CI=[0.740, 0.885], mean=10.421 ms, p95=25.833 ms
depth=5: 79W/4D/17L, score=0.810, CI=[0.740, 0.875], mean=43.638 ms, p95=124.390 ms
depth=6: 80W/2D/18L, score=0.810, CI=[0.735, 0.880], mean=148.714 ms, p95=439.879 ms


![A5: độ sâu Alpha-Beta](../results/figures/ablation_a5_search.png)


## Giới hạn và tái lập

- Trung bình/p95 tính trên **từng nước** của tác tử khảo sát, không lấy trung bình các p95 theo ván.
- Tất cả ván đã được phát lại bằng ReferenceGame; chương trình dừng khi sai kết quả hoặc ngân sách mô phỏng.
- Một bộ 50 khai cuộc và một lịch seed; chưa có các cohort khai cuộc độc lập để suy rộng.
- Thời gian phụ thuộc máy và tải nền; không dùng số nút hoặc số mô phỏng thay cho giây khi tuyên bố công bằng ngân sách.
- Chi tiết cấu hình, commit, môi trường và dấu thời gian ở `results/logs/run_manifest.csv`.
- Đây là dữ liệu đo tại lần chạy ghi trong manifest; chưa tự coi là bộ đóng băng ngày 19/11.
